# 🔴 Solution: KV Cache Attention（参考版）

## 📋 题目描述

**难度：** Hard

实现带 KV 缓存的多头注意力，用于高效自回归解码。

KV 缓存存储之前计算的键和值，使每个解码步骤只需处理新 token 而非完整序列。

**签名:** `KVCacheAttention(d_model, num_heads)`（nn.Module）

**前向传播:** `forward(x, cache=None) -> (Tensor, tuple)`
- `x` — 输入张量 (B, S_new, d_model)
- `cache` — 可选的前序步骤 (K, V) 元组

**返回:** (输出, (K_all, V_all))，缓存逐步增长

**约束:**
- 将新 K/V 与缓存沿序列维度拼接
- 预填充时（S_new > 1）应用因果掩码
- 增量解码必须与完整前向传播数值一致

**提示：** 1. 投影 Q/K/V → reshape 为 `(B, num_heads, S, d_k)`
2. 有缓存时：K = cat([cache_K, K], dim=2),  V = cat([cache_V, V], dim=2)
3. S_new > 1（预填充）时应用因果掩码
4. 缩放点积注意力 → 输出投影
5. 返回 (output, (K_all, V_all))


In [ ]:
import torch
import math

In [ ]:
# ✅ SOLUTION

def attention_with_kv_cache(x: torch.Tensor, W_q: torch.Tensor, W_k: torch.Tensor, W_v: torch.Tensor, W_o: torch.Tensor, num_heads: int, cached_key: torch.Tensor = None, cached_value: torch.Tensor = None, mask: torch.Tensor = None) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    # KV Cache：自回归生成时的推理优化
    # 问题：生成每个 token 时都要重新计算所有 token 的 K, V → 重复计算
    # 解决：缓存已计算的 K, V，每次只计算新 token 的 Q, K, V

    batch_size, seq_len, d_model = x.shape
    d_k = d_model // num_heads

    # ---- Step 1: 计算当前 token 的 QKV ----
    Q = x @ W_q
    K_new = x @ W_k
    V_new = x @ W_v

    # ---- Step 2: 拆分多头 ----
    Q = Q.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)
    K_new = K_new.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)
    V_new = V_new.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)

    # ---- Step 3: 拼接缓存 ----
    # torch.cat 沿 seq 维度拼接：旧 KV + 新 KV
    # 首次调用时 cached 为 None，直接用新的
    if cached_key is not None:
        K = torch.cat([cached_key, K_new], dim=2)
        V = torch.cat([cached_value, V_new], dim=2)
    else:
        K = K_new
        V = V_new

    # ---- Step 4: 注意力计算 ----
    # Q 只有新 token 的 query，但 K, V 包含所有历史
    # scores shape: [B, H, new_len, total_len]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    attn = torch.softmax(scores, dim=-1)
    out = attn @ V

    # ---- Step 5: 合并 + 投影 ----
    out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
    return out @ W_o, K, V

In [ ]:
# Demo: full forward vs incremental decode
torch.manual_seed(0)
attn = KVCacheAttention(d_model=64, num_heads=4)
x = torch.randn(1, 6, 64)

full_out, _ = attn(x)
out1, cache = attn(x[:, :4])
out2, cache = attn(x[:, 4:5], cache=cache)
out3, cache = attn(x[:, 5:6], cache=cache)
inc_out = torch.cat([out1, out2, out3], dim=1)

print('Full shape:', full_out.shape)
print('Match:', torch.allclose(full_out, inc_out, atol=1e-5))
print('Final cache K shape:', cache[0].shape)

## 📝 核心思路总结

1. **KV Cache 核心**：缓存历史 K, V，每次只计算新 token 的投影
2. **torch.cat 拼接**：沿 seq 维度拼接新旧 KV
3. **推理优化**：避免重复计算，生成复杂度 O(n²) → O(n)
4. **内存权衡**：用空间换时间，需注意 KV cache 内存管理